# Sloane 273 Revelation: PDF to OSIS

Converts Nehemia Gordon's transcription and translation of British Library MS Sloane 273 into clean Hebrew, annotated Hebrew, and annotated English OSIS. The supplied PDF contains Revelation 1:1-2:13.

In [ ]:
from __future__ import annotations

from collections import OrderedDict
import sys
from pathlib import Path
import re
import unicodedata

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import fitz  # PyMuPDF
from lxml import etree

PDF_PATH = Path('../data/00_source_files/A-Hebrew-Manuscript-of-the-Book-of-Revelation-British-Library-Sloane-273.pdf')
OUT_DIR = Path('../data/01_osis')
STEM = 'REV_Sloane273'
OSIS_NS = 'http://www.bibletechnologies.net/2003/OSIS/namespace'
XSI_NS = 'http://www.w3.org/2001/XMLSchema-instance'
SCHEMA = Path('pdf2osis/schema/osisCore.2.1.1-project-subset.xsd')

assert PDF_PATH.exists(), PDF_PATH
assert SCHEMA.exists(), SCHEMA
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Input: {PDF_PATH} ({len(fitz.open(PDF_PATH))} pages)')

In [ ]:
# The PDF stores Hebrew in visual order. Reverse each extracted line, not the
# whole page, so multi-line text keeps its printed top-to-bottom order.
HEBREW_RE = re.compile(r'[\u0590-\u05ff]')
VERSE_MARKER_RE = re.compile(r'\[\s*(\d+)\s*\]')
ENGLISH_MARKER_RE = re.compile(r'(?<![A-Za-z0-9])(\d+)\s*[.)]\s+(?=[A-Z])')

def normalise(text: str) -> str:
    text = unicodedata.normalize('NFC', text)
    # A combining mark cannot begin a Hebrew word; discard only such orphaned
    # glyphs from the PDF text layer (positioned Hebrew uses rtl_line above).
    text = re.sub(r'(?<!\S)[\u0591-\u05c7]+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def rtl_line(spans: list[dict]) -> str:
    """Rebuild a Hebrew line from positioned glyphs without reversing marks."""
    chars = [char for span in spans for char in span.get('chars', [])]
    bases = [
        {'text': char['c'], 'x': (char['bbox'][0] + char['bbox'][2]) / 2, 'marks': []}
        for char in chars if not unicodedata.combining(char['c'])
    ]
    if not bases:
        return ''
    for char in chars:
        mark = char['c']
        if unicodedata.combining(mark):
            x = (char['bbox'][0] + char['bbox'][2]) / 2
            letter_bases = [base for base in bases if base['text'].isalpha()]
            nearest = min(letter_bases or bases, key=lambda base: abs(base['x'] - x))
            nearest['marks'].append(mark)
    # Hebrew glyphs are printed right-to-left; sort bases by x descending and
    # append each glyph's combining marks in Unicode canonical order.
    text = ''.join(
        base['text'] + ''.join(sorted(base['marks'], key=unicodedata.combining))
        for base in sorted(bases, key=lambda base: base['x'], reverse=True)
    )
    return normalise(text)

def page_lines(page, *, side: str):
    lines = []
    for block in page.get_text('rawdict')['blocks']:
        if block.get('type') != 0:
            continue
        x0, y0, x1, y1 = block['bbox']
        # Main text is above the page footnotes and below the running header.
        if y0 < 60 or y1 > 555:
            continue
        for line in block.get('lines', []):
            spans = line.get('spans', [])
            spans = [s for s in spans if (s['bbox'][0] >= 310 if side == 'hebrew' else s['bbox'][2] <= 310)]
            raw = ''.join(char['c'] for span in spans for char in span.get('chars', []))
            text = rtl_line(spans) if side == 'hebrew' else normalise(raw)
            # In the visual Hebrew text layer, a verse marker is rendered as
            # a bracketed number interwoven with nearby Hebrew glyphs. The
            # original span still retains the bracket, unlike superscripts.
            numbers = re.findall(r'\d+', raw) if side == 'hebrew' and '[' in raw else []
            verse_marker = int(numbers[0]) if numbers else None
            if text:
                lines.append((line['bbox'][1], text, verse_marker))
    return sorted(lines, key=lambda item: item[0])

def page_footnotes(page) -> dict[str, str]:
    chunks = []
    for block in page.get_text('blocks'):
        if 500 <= block[1] < 650 and block[4].strip():
            chunks.append(block[4])
    text = normalise(' '.join(chunks))
    starts = list(re.finditer(r'(?<!\S)(\d+)\s+', text))
    notes = {}
    for index, match in enumerate(starts):
        end = starts[index + 1].start() if index + 1 < len(starts) else len(text)
        notes[match.group(1)] = text[match.end():end].strip()
    return notes

def clean_hebrew(text: str) -> str:
    text = re.sub(r'\][\s\u0590-\u05ff]{1,4}(?=\s|\[)', ' ', text)
    text = re.sub(r'\[[^]]*\]', ' ', text)  # manuscript folio labels
    text = text.replace('[', ' ').replace(']', ' ')
    text = re.sub(r'\d+', ' ', text)          # note markers are encoded separately
    return normalise(text)

def append(record, field: str, text: str):
    text = normalise(text)
    if text:
        record[field] = normalise(f"{record[field]} {text}")

def key(chapter: int, verse: int) -> tuple[int, int]:
    return chapter, verse

In [ ]:
# Extract right-column Hebrew first. Its bracketed verse numbers are the
# authoritative anchors for pairing the adjacent English column.
records: OrderedDict[tuple[int, int], dict] = OrderedDict()
current_chapter = 1
current_key = None
expected_verse = 1

with fitz.open(PDF_PATH) as document:
    for page_number, page in enumerate(document, start=1):
        if page_number == 1:
            continue  # title/front matter, including its non-verse footnote 1
        starts = []
        page_notes = page_footnotes(page)
        previous_key = current_key

        hebrew_lines = page_lines(page, side='hebrew')
        first_marker = next((marker for _, _, marker in hebrew_lines if marker is not None), None)
        if first_marker is not None and hebrew_lines and hebrew_lines[0][2] is None:
            first_candidates = {first_marker, int(str(first_marker)[::-1])}
            if expected_verse + 1 in first_candidates:
                # The unnumbered top continuation belongs to the intervening verse.
                current_key = key(current_chapter, expected_verse)
                records.setdefault(current_key, {'chapter': current_chapter, 'verse': expected_verse, 'hebrew': '', 'english': '', 'notes': OrderedDict(), 'pages': set()})
                records[current_key]['pages'].add(page_number)
                starts.append((60, current_key))
                expected_verse += 1
        for y, raw_line, detected_marker in hebrew_lines:
            if detected_marker is not None:
                candidates = {detected_marker, int(str(detected_marker)[::-1])}
                if expected_verse in candidates:
                    verse = expected_verse
                elif expected_verse + 1 in candidates:
                    # Rev. 1:18 has no surviving numeral in the PDF text layer.
                    # Retain its source position rather than merging it into 1:17.
                    missing_key = key(current_chapter, expected_verse)
                    records.setdefault(missing_key, {'chapter': current_chapter, 'verse': expected_verse, 'hebrew': '', 'english': '', 'notes': OrderedDict(), 'pages': {page_number}})
                    starts.append((60, missing_key))
                    expected_verse += 1
                    verse = expected_verse
                else:
                    verse = None
                if verse is not None:
                    current_key = key(current_chapter, verse)
                    records.setdefault(current_key, {
                        'chapter': current_chapter, 'verse': verse, 'hebrew': '',
                        'english': '', 'notes': OrderedDict(), 'pages': set(),
                    })
                    records[current_key]['pages'].add(page_number)
                    starts.append((y, current_key))
                    expected_verse = verse + 1
            if current_key is not None:
                segment = raw_line
                append(records[current_key], 'hebrew', segment)
                # Note glyphs are visually reversed in the text layer; their
                # audited source locations are applied below.

        # Pair English lines with the most recent Hebrew start on the page.
        # A small tolerance handles the PDF's differently aligned columns.
        page_current = previous_key
        for y, english_line, _ in page_lines(page, side='english'):
            candidates = [item for item in starts if item[0] <= y + 12]
            if candidates:
                page_current = candidates[-1][1]
            if page_current is None:
                continue
            cursor = 0
            for marker in ENGLISH_MARKER_RE.finditer(english_line):
                append(records[page_current], 'english', english_line[cursor:marker.start()])
                candidate = key(current_chapter, int(marker.group(1)))
                if candidate in records:
                    page_current = candidate
                cursor = marker.end()
            append(records[page_current], 'english', english_line[cursor:])

        # The chapter heading is printed after Rev. 1:20 on page 7.
        chapter_match = re.search(r'Revelation\s*-\s*Chapter\s+(\d+)', page.get_text())
        if chapter_match:
            current_chapter = int(chapter_match.group(1))
            expected_verse = 1

# The PDF uses small, sometimes poorly extracted superscripts. These source
# locations ensure every verse-linked footnote survives even when its glyph
# is absent from the text layer. Footnote 1 belongs to the front matter.
fallback_note_targets = {
    '2': (1, 1), '3': (1, 2), '4': (1, 5), '5': (1, 7), '6': (1, 10),
    '7': (1, 15), '8': (1, 20), '9': (2, 2), '10': (2, 2), '11': (2, 2),
    '12': (2, 5), '13': (2, 7), '14': (2, 9), '15': (2, 12),
}
with fitz.open(PDF_PATH) as document:
    all_notes = {}
    for page in document:
        all_notes.update(page_footnotes(page))
for number, target in fallback_note_targets.items():
    if number in all_notes:
        records[target]['notes'][number] = all_notes[number]

expected = [key(1, verse) for verse in range(1, 21)] + [key(2, verse) for verse in range(1, 14)]
assert list(records) == expected, f'Unexpected coverage: {list(records)}'
for record in records.values():
    record['hebrew'] = clean_hebrew(record['hebrew'])
    record['english'] = normalise(re.sub(r'^\(?\d+\)?\s*', '', record['english']))

print(f'Extracted {len(records)} verses: Rev.1.1 through Rev.2.13')
print(f'Footnotes linked to verses: {sum(len(record["notes"]) for record in records.values())}')

In [ ]:
def tag(name: str) -> str:
    return f'{{{OSIS_NS}}}{name}'

def make_header(parent, *, work_id: str, language: str, title: str, translation: bool = False):
    header = etree.SubElement(parent, tag('header'))
    work = etree.SubElement(header, tag('work'), osisWork=work_id)
    etree.SubElement(work, tag('title')).text = title
    etree.SubElement(work, tag('identifier'), type='OSIS').text = work_id
    etree.SubElement(work, tag('scope')).text = 'REV'
    etree.SubElement(work, tag('language')).text = language
    etree.SubElement(work, tag('type'), type='x-manuscript' if not translation else 'x-translation').text = 'Manuscript' if not translation else 'Translation'
    etree.SubElement(work, tag('identifier'), type='shelfmark').text = 'British Library, MS Sloane 273'
    etree.SubElement(work, tag('contributor'), role='trl' if translation else 'trc', **{'file-as': 'Gordon, Nehemia'}).text = 'Nehemia Gordon'
    etree.SubElement(work, tag('date'), event='eversion', type='ISO').text = '2017'
    etree.SubElement(work, tag('rights')).text = '© 2017 Nehemia Gordon'
    etree.SubElement(work, tag('description')).text = ('Hebrew transcription and English translation of the Revelation passages in British Library MS Sloane 273, transcribed and translated by Nehemia Gordon.')
    bible = etree.SubElement(header, tag('work'), osisWork='bible')
    etree.SubElement(bible, tag('identifier'), type='OSIS').text = 'bible'
    etree.SubElement(bible, tag('refSystem')).text = 'StandardV11N'
    etree.SubElement(bible, tag('language')).text = language

def build_osis(variant: str):
    is_translation = variant == 'translation'
    commented = variant == 'hebrew_commented'
    work_id = {
        'hebrew': 'REV_Sloane273_Hebrew',
        'hebrew_commented': 'REV_Sloane273_Hebrew_Commented',
        'translation': 'REV_Sloane273_Translation',
    }[variant]
    language = 'en' if is_translation else 'he'
    title = 'English Translation of Revelation (British Library MS Sloane 273)' if is_translation else 'Revelation (British Library MS Sloane 273)'
    root = etree.Element(tag('osis'), nsmap={None: OSIS_NS, 'xsi': XSI_NS})
    root.set(f'{{{XSI_NS}}}schemaLocation', f'{OSIS_NS} http://www.bibletechnologies.net/osisCore.2.1.1.xsd')
    text = etree.SubElement(root, tag('osisText'), osisIDWork=work_id, osisRefWork='bible')
    text.set('{http://www.w3.org/XML/1998/namespace}lang', language)
    make_header(text, work_id=work_id, language=language, title=title, translation=is_translation)
    book = etree.SubElement(text, tag('div'), type='book', osisID='Rev')
    chapter = None
    for record in records.values():
        if record['chapter'] != chapter:
            chapter = record['chapter']
            chapter_el = etree.SubElement(book, tag('chapter'), osisID=f'Rev.{chapter}')
        verse_el = etree.SubElement(chapter_el, tag('verse'), osisID=f"Rev.{record['chapter']}.{record['verse']}", n=str(record['verse']))
        verse_el.text = record['english'] if is_translation else record['hebrew']
        if commented or is_translation:
            for number, note_text in record['notes'].items():
                note = etree.SubElement(verse_el, tag('note'), type='footnote', n=number)
                note.text = note_text
                note.tail = ' '
    return etree.ElementTree(root)

trees = {variant: build_osis(variant) for variant in ('hebrew', 'hebrew_commented', 'translation')}
outputs = {variant: OUT_DIR / f'{STEM}_{variant}.osis' for variant in trees}
for variant, tree in trees.items():
    tree.write(outputs[variant], encoding='UTF-8', xml_declaration=True, pretty_print=True)
    print(f'Wrote {outputs[variant]}')

In [ ]:
# Verification: XML/schema validity, unique ordered coverage, variants, and
# human-readable spot checks for the opening, a cross-page verse, and notes.
schema = etree.XMLSchema(etree.parse(str(SCHEMA)))
namespace = {'osis': OSIS_NS}
for variant, path in outputs.items():
    tree = etree.parse(str(path))
    assert schema.validate(tree), schema.error_log
    ids = tree.xpath('//osis:verse/@osisID', namespaces=namespace)
    assert ids == [f'Rev.{chapter}.{verse}' for chapter, verse in expected]
    assert len(ids) == len(set(ids)) == 33
    notes = tree.xpath('//osis:note', namespaces=namespace)
    if variant == 'hebrew':
        assert not notes
    else:
        assert len(notes) == 14
    print(f'{path.name}: {len(ids)} verses, {len(notes)} notes, schema valid')

assert records[(1, 1)]['hebrew']
assert records[(1, 18)]['hebrew'] and records[(1, 18)]['pages'] == {7}
assert records[(2, 2)]['notes']['9']
print('Rev 1:1 Hebrew:', records[(1, 1)]['hebrew'][:100])
print('Rev 1:18 continuation page:', sorted(records[(1, 18)]['pages']))
print('Footnote 9:', records[(2, 2)]['notes']['9'])